# Ce notebook à vocation à tester la pertinence de la méthode des KNN pour notre problématique 

Il s'agit d'un notebook exploratoire sur cette méthode. 
Le but est de la comparer avec nos autres modèles/méthodes pour identifier la plus pertinente puis de travailler cette dernière en profondeur.
Ainsi si le choix du KNN s'impose, le socle présent dans ce notebook peut être ammené à être affiné dans le .py final. 

## Import des fonctions et du dataset

In [1]:
import mlflow
import numpy as np
from mlflow.models import infer_signature
from sklearn.compose import ColumnTransformer
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder, StandardScaler

from base_commune.outils import (
    RANDOM_STATE,
    charger_donnees,
    configurer_mlflow,
    decouper_donnees,
    evaluer_modele,
    exporter_runs,
    tags_reproductibilite,
)

In [2]:
df_brut = charger_donnees()
df_brut.shape
df_brut.head(5)

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


In [3]:
X_train, X_test, y_train, y_test = decouper_donnees(df_brut)
print(X_train.shape)
print(X_test.shape)
# Note : comme mentionné dans les .md on va travailler sur de la cross validation pour le test, ici le découpage train/test via la fonction nous garantit juste qu'on conserve 20%
# du dataset pour un vrai test final

(8000, 10)
(2000, 10)


## Préparation des données
prépa du pipeline de transfo des données pour les variables qui en ont besoins

In [4]:
vars_quanti = ["CreditScore", "Tenure", "Balance", "EstimatedSalary"]
vars_cate = ["Geography", "Gender", "NumOfProducts"]
vars_bin = ["HasCrCard", "IsActiveMember"]
vars_age = ["Age"]

colonnes_modele = vars_quanti + vars_cate + vars_bin + vars_age
assert set(colonnes_modele) == set(X_train.columns), "variable oubliée ou en trop"

def creer_tranches_age(X):
    # Conversion directe en tableau NumPy numérique
    if hasattr(X, "to_numpy"):
        ages = X.to_numpy(dtype=np.float64, na_value=np.nan).ravel()
    else:
        ages = np.asarray(X, dtype=np.float64).ravel()

    # Bornes des 8 tranches
    bornes = np.array([28, 38, 48, 58, 68, 78, 88])

    labels = np.array([
        "18-27",
        "28-37",
        "38-47",
        "48-57",
        "58-67",
        "68-77",
        "78-87",
        "88-97"
    ], dtype=object)

    # Création des catégories sans opération sur les chaînes Pandas
    indices = np.digitize(ages, bornes)

    result = labels[indices].astype(object)

    # Gestion des éventuelles valeurs manquantes
    result[np.isnan(ages)] = "Age_inconnu"
    return result.reshape(-1, 1)

pipeline_quanti = make_pipeline(StandardScaler())
pipeline_cate = make_pipeline(OneHotEncoder(handle_unknown="ignore"))
pipeline_age = make_pipeline(FunctionTransformer(creer_tranches_age, validate=False),OneHotEncoder(handle_unknown="ignore"))


preprocesseur = ColumnTransformer(
    transformers=[
        ("quanti", pipeline_quanti, vars_quanti),
        ("cate", pipeline_cate, vars_cate),
        ("age", pipeline_age, vars_age)],remainder="drop")



X = X_train[vars_quanti + vars_cate + vars_age]
X_transformee = preprocesseur.fit_transform(X)

print("ca marche")

ca marche


In [5]:
def construire_pipeline(n_neighbors=5, weights="distance", age_en_tranches=False, retirer=()):
    if age_en_tranches:
        transfo_age = make_pipeline(
            FunctionTransformer(creer_tranches_age, validate=False),
            OneHotEncoder(handle_unknown="ignore"))
    else:
        transfo_age = StandardScaler()

    preprocesseur = ColumnTransformer([
        ("quanti", StandardScaler(), [v for v in vars_quanti if v not in retirer]),
        ("cate", OneHotEncoder(handle_unknown="ignore"), vars_cate),
        ("bin", "passthrough", vars_bin),
        ("age", transfo_age, vars_age),
    ], remainder="drop")

    return Pipeline([("preprocesseur", preprocesseur),
                     ("knn", KNeighborsClassifier(n_neighbors=n_neighbors, weights=weights))])

In [6]:
configurer_mlflow("churn_bancaire")

def lancer_run(n_neighbors, weights, age_en_tranches):
    pipeline = construire_pipeline(n_neighbors, weights, age_en_tranches)
    codage = "age_tranches" if age_en_tranches else "age_num"
    with mlflow.start_run(run_name=f"knn_k{n_neighbors}_{weights}_{codage}"):
        mlflow.set_tags({"model_family": "knn", "author": "roman",
                         "stage": "exploration", **tags_reproductibilite()})
        mlflow.log_params({"n_neighbors": n_neighbors, "weights": weights,
                           "age_en_tranches": age_en_tranches,
                           "features": ",".join(colonnes_modele),
                           "cv_folds": 5, "random_state": RANDOM_STATE,
                           "test_size": 0.2, "n_train": len(X_train)})
        metriques = evaluer_modele(pipeline, X_train, y_train)
        mlflow.log_metrics(metriques)

        pipeline.fit(X_train, y_train)
        signature = infer_signature(X_train, pipeline.predict_proba(X_train)[:, 1])
        mlflow.sklearn.log_model(pipeline, name="model", signature=signature,
                                 input_example=X_train.head(3),
                                 serialization_format="cloudpickle")
    return metriques

On va tester le meilleur combo en lancant cette petite boucle :

In [7]:
for k in [5, 15, 31, 51]:
    for poids in ["uniform", "distance"]:
        for tranches in [True, False]:
            m = lancer_run(k, poids, tranches)
            print(k, poids, tranches, round(m["cv_pr_auc_mean"], 3))

c:\Users\33698\Desktop\Cours\Master\M2 DS SIAD\projet_mlops\mlops_roman_delbare\.venv\Lib\site-packages\mlflow\types\utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(
2026/10/31 00:10:42 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object ser

5 uniform True 0.465


c:\Users\33698\Desktop\Cours\Master\M2 DS SIAD\projet_mlops\mlops_roman_delbare\.venv\Lib\site-packages\mlflow\types\utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(
2026/10/31 00:11:02 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object ser

5 uniform False 0.478


c:\Users\33698\Desktop\Cours\Master\M2 DS SIAD\projet_mlops\mlops_roman_delbare\.venv\Lib\site-packages\mlflow\types\utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(
2026/10/31 00:11:16 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object ser

...

51 uniform False 0.589


c:\Users\33698\Desktop\Cours\Master\M2 DS SIAD\projet_mlops\mlops_roman_delbare\.venv\Lib\site-packages\mlflow\types\utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(
2026/10/31 00:14:11 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object ser

51 distance True 0.567


c:\Users\33698\Desktop\Cours\Master\M2 DS SIAD\projet_mlops\mlops_roman_delbare\.venv\Lib\site-packages\mlflow\types\utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(
2026/10/31 00:14:26 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object ser

51 distance False 0.603


In [8]:
chemin = exporter_runs("churn_bancaire", "roman")
print(chemin)

C:\Users\33698\Desktop\Cours\Master\M2 DS SIAD\projet_mlops\mlops_roman_delbare\reports\runs_roman.csv


Sur la meilleure version on a un PR-AUC d’environ 0,60 soit environ 3 fois le hasard (0,204). ROC-AUC de 0,83 et accuracy de 0,834 contre 0,796 pour un modèle qui prédit toujours « reste ». Le modèle est donc relativement correct pour une problématique de churn. L'augmentation du nombre de voisins augmente les perfs du modèle sur notre métrique cible ici (PR-AUC) mais pénalise parcontre le recall fortement et par conséquent le F-1 Score (logique comme c une moyenne harmonique en partie basée dessus). Cependant c'est relativement "normal" car ici on garde toujours 0.5 comme seuil de décision. C'est surement un effet du seuil de décision : le modèle classe mieux les clients mais ne prédit
« départ » que lorsque la majorité des voisins part, ce qui est rare avec 20 % de churners. Comme mentionné précedemment, si on conserve le KNN comme modèle on cherchera ensuite à l'optimiser et notamment le seuil de précision de ce dernier pour corriger ce défaut.

On va maintenant essayer d'uagmenter encore k, et de tester des versions sans les variables qui n'apportent visiblement (cf stats desc) presque que du bruit pour voir : 

In [10]:
def lancer_run(n_neighbors, weights="distance", age_en_tranches=False, retirer=()):
    pipeline = construire_pipeline(n_neighbors, weights, age_en_tranches, retirer)
    utilisees = [c for c in colonnes_modele if c not in retirer]
    codage = "age_tranches" if age_en_tranches else "age_num"
    suffixe = "_sans_" + "_".join(retirer) if retirer else ""
    with mlflow.start_run(run_name=f"knn_k{n_neighbors}_{weights}_{codage}{suffixe}"):
        mlflow.set_tags({"model_family": "knn", "author": "roman",
                         "stage": "exploration", **tags_reproductibilite()})
        mlflow.log_params({"n_neighbors": n_neighbors, "weights": weights,
                           "age_en_tranches": age_en_tranches,
                           "features": ",".join(utilisees),
                           "retirees": ",".join(retirer) or "aucune",
                           "cv_folds": 5, "random_state": RANDOM_STATE,
                           "test_size": 0.2, "n_train": len(X_train)})
        metriques = evaluer_modele(pipeline, X_train, y_train)
        mlflow.log_metrics(metriques)

        pipeline.fit(X_train, y_train)
        signature = infer_signature(X_train, pipeline.predict_proba(X_train)[:, 1])
        mlflow.sklearn.log_model(pipeline, name="model", signature=signature,
                                 input_example=X_train.head(3),
                                 serialization_format="cloudpickle")
    return metriques

In [12]:
for k in [51, 101, 151]:
    for retirer in [(), ("Tenure", "EstimatedSalary"),
                    ("Tenure", "EstimatedSalary", "CreditScore")]:
        m = lancer_run(k, "distance", False, retirer)
        print(k, retirer, round(m["cv_pr_auc_mean"], 3))

chemin = exporter_runs("churn_bancaire", "roman")

c:\Users\33698\Desktop\Cours\Master\M2 DS SIAD\projet_mlops\mlops_roman_delbare\.venv\Lib\site-packages\mlflow\types\utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(
2026/10/31 00:42:31 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object ser

51 () 0.603


c:\Users\33698\Desktop\Cours\Master\M2 DS SIAD\projet_mlops\mlops_roman_delbare\.venv\Lib\site-packages\mlflow\types\utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(
2026/10/31 00:42:56 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object ser

51 ('Tenure', 'EstimatedSalary') 0.621


c:\Users\33698\Desktop\Cours\Master\M2 DS SIAD\projet_mlops\mlops_roman_delbare\.venv\Lib\site-packages\mlflow\types\utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(
2026/10/31 00:43:18 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object ser

...

151 () 0.587


c:\Users\33698\Desktop\Cours\Master\M2 DS SIAD\projet_mlops\mlops_roman_delbare\.venv\Lib\site-packages\mlflow\types\utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(
2026/10/31 00:45:17 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object ser

151 ('Tenure', 'EstimatedSalary') 0.615


c:\Users\33698\Desktop\Cours\Master\M2 DS SIAD\projet_mlops\mlops_roman_delbare\.venv\Lib\site-packages\mlflow\types\utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(
2026/10/31 00:45:45 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object ser

151 ('Tenure', 'EstimatedSalary', 'CreditScore') 0.55


- Au-delà de k = 51, aucun gain (PR-AUC en légère baisse, recall en forte baisse). CA sembloe logique carr on perd toute finesse
- Retirer Tenure et EstimatedSalary (variables sans lien visible avec le churn) améliore légèrement et de façon constante les résultats. 
- Retirer aussi CreditScore dégrade nettement le modèle . Je pense qu'elle va très bien s'intégrer sur un modèle à base d'arbre
- Le F1 au seuil de 0,5 ne sert pas de critère de choix : la variante sans les trois variables a le meilleur F1 mais la pire PR-AUC. En plus comme dt précédemment on à ça car on à pas opti le seuil et qu'on fait exploser k.

DONC =
Configuration retenue : k = 51, pondération par la distance, âge numérique, sans Tenure ni EstimatedSalary 
On va tester d'autres modèles et on verra si c'est cette config de KNN qu'on optimise ou non.